# ⚡ Crushing Stiffness: The Deterministic PINNsFormer
Welcome to the deterministic foundation of the **PINNsFormer**. Before we can measure the uncertainty of a model, we must ensure the core physics engine is flawless. 

In this notebook, we are abandoning standard Multi-Layer Perceptrons (MLPs) and numerical ODE solvers. Instead, we are utilizing **Sequence-to-Sequence (Seq2Seq)** modeling and **Wavelet Activations** to conquer the stiffness of the Hodgkin-Huxley (HH) equations.

---

## 1. The Data Paradigm Shift: Time as a Sequence
Standard Neural ODEs process time sequentially: $u_{t+1} = u_t + \text{Solver}(f(u))$. If the system is stiff (like an action potential), numerical errors compound at every step.

The PINNsFormer completely changes how we view time. We stop stepping forward and instead process the entire temporal trajectory as a single, holistic object.

To do this, we must reshape our data to fit the standard Transformer tensor format:
$$ \text{Shape: } (\text{Features}, \text{Sequence Length}, \text{Batch Size}) $$

By treating the entire 100ms simulation as a single sequence batch, the network can look at the past, present, and future simultaneously.

## 2. The Architecture: Attention and Wavelets
Standard Physics-Informed Neural Networks (PINNs) suffer from **Spectral Bias**—they naturally prefer learning low-frequency, smooth functions. The HH equations, however, are dominated by high-frequency, violent voltage spikes. If a standard MLP tries to learn this, it will either smooth out the spike or oscillate uncontrollably. 

We solve this using a two-pronged architectural approach:

### A. Global Temporal Awareness (Self-Attention)
The Transformer's Self-Attention mechanism computes how every single time point relates to every other time point in the sequence:
$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$
This allows the network to instantly correlate the slow buildup of the gating variables at $t=10$ms with the explosive voltage spike at $t=35$ms, entirely bypassing the need for an ODE solver.

### B. The Anti-Stiffness Weapon (Mexican Hat Wavelet)
To defeat Spectral Bias, we replace standard global activations (like $\tanh$) inside the Feed-Forward block with the **Mexican Hat Wavelet**:
$$ \psi(x) = (1 - x^2) \exp\left(-\frac{x^2}{2}\right) $$
Unlike $\tanh$, which affects the output everywhere, a wavelet is localized. It stays near zero during the flat resting potential, and sharply "fires" only when the HH dynamics demand an action potential. The network learns to stretch and shift these wavelets to perfectly capture high-frequency stiff dynamics without corrupting the smooth areas.

## 3. The Seq2Seq Physics Loss (Finite Differences)
Because we have eliminated the ODE solver, we can no longer evaluate the physics at a single point in time and step forward. We must enforce the Hodgkin-Huxley physics directly across the entire predicted sequence at once.

### The Physics Residual
We need to know the predicted temporal derivative ($du/dt$) of our sequence. Since doing continuous auto-differentiation through complex Attention layers is computationally brutal, we use **Central Finite Differences** across our discrete sequence $U_{pred}$:
$$ \frac{d u_i}{dt} \approx \frac{u_{i+1} - u_{i-1}}{2\Delta t} $$

We pass our interior sequence points through the exact HH physical equations—let's call that mathematical operator $\mathcal{F}(u)$—and penalize any difference between our sequence's actual derivative and the theoretical derivative:
$$ \mathcal{L}_{phys} = \frac{1}{N} \sum_{i=2}^{N-1} \left\| \frac{u_{i+1} - u_{i-1}}{2\Delta t} - \mathcal{F}(u_i) \right\|^2 $$

*Note: Just like in the PI-NODE-SR framework, we still divide this residual by our Scale Factors ($s_j$) to ensure the massive voltage gradients do not drown out the tiny gating gradients!*

## 4. The Optimization Engine
We now optimize the deterministic network weights $\theta$ to minimize our combined objective:
$$ \mathcal{L}_{total} = \mathcal{L}_{data} + \lambda \mathcal{L}_{phys} $$

### Why this is wildly efficient:
In your previous Neural ODE setup, the automatic differentiator (`Zygote`) had to backpropagate gradients *through* the complex operations of a differential equation solver. 

Here, Zygote has a much easier job. It simply differentiates the Transformer's network parameters directly relative to the algebraic loss function. 

*Transformer Training Tip: Attention-based models can be sensitive to large gradient steps early in training. We lower the Adam optimizer's learning rate slightly (e.g., to $0.0005$) to ensure the Self-Attention weights stabilize smoothly.*

In [ ]:
# %%
using Lux, SciMLSensitivity, Optimization, OptimizationOptimisers, Statistics, Random, ComponentArrays, Zygote

# Assuming df_ordered, t_train, z_train, and scale_factors are already in memory 
# from your previous data processing steps.

# Transformers expect data in shape: (feature_dim, seq_len, batch_size)
# Here, our entire trajectory is 1 sequence batch.
t_seq = reshape(t_train, 1, length(t_train), 1) # Shape: (1, N, 1)
z_seq = reshape(z_train, 4, length(t_train), 1) # Shape: (4, N, 1)

# Time step size for finite difference physics derivatives
dt = Float32(t_train[2] - t_train[1])

rng = Random.default_rng()
Random.seed!(rng, 42)

In [ ]:
# %% 
# 1. Mexican Hat Wavelet Activation
mexican_hat(x) = (1.0f0 .- x.^2) .* exp.(-0.5f0 .* x.^2)

# 2. Custom Self-Attention Wrapper for Lux
# Lux's MHA needs (Q, K, V). For self-attention, Q = K = V = x.
struct SelfAttention{M} <: Lux.AbstractExplicitLayer
    mha::M
end
SelfAttention(d_model, n_heads) = SelfAttention(Lux.MultiHeadAttention(d_model, n_heads))

function ((layer::SelfAttention)(x, ps, st::NamedTuple))
    # Pass x as Query, Key, and Value
    # MHA returns (output, attention_weights), we just want the output
    out, st_mha = layer.mha((x, x, x), ps, st)
    return out[1], st_mha 
end

# 3. Transformer Hyperparameters
d_model = 32     # Hidden dimension size
n_heads = 4      # Number of attention heads
d_ff = 64        # Feed-forward network dimension

# 4. Build the Deterministic PINNsFormer
pinnsformer = Lux.Chain(
    # A. Time Embedding: Map 1D time (t) to d_model dimensional space
    Lux.Dense(1 => d_model),
    
    # B. Self-Attention Block with Residual Connection
    Lux.SkipConnection(
        Lux.Chain(
            Lux.LayerNorm((d_model,)),
            SelfAttention(d_model, n_heads)
        ),
        +
    ),
    
    # C. Wavelet Feed-Forward Block (Deterministic, NO Dropout)
    Lux.SkipConnection(
        Lux.Chain(
            Lux.LayerNorm((d_model,)),
            Lux.Dense(d_model => d_ff),
            Lux.WrappedFunction(mexican_hat), # The anti-stiffness secret weapon
            Lux.Dense(d_ff => d_model)
        ),
        +
    ),
    
    # D. Output Projection: Map back to the 4 HH states (V, n, m, h)
    Lux.Dense(d_model => 4)
)

# Initialize network weights and state on CPU
ps, st = Lux.setup(rng, pinnsformer)
p_nn = ComponentArray(ps)

In [ ]:
# %%
# (Assumes your hh_equations(u) function is defined exactly as before)

function physics_loss_seq(u_pred, dt_step, s_factors)
    # u_pred is shape: (4, N, 1)
    
    # 1. Approximate du/dt using central finite differences on the sequence
    # Drop first and last points to calculate symmetric derivatives
    u_next = u_pred[:, 3:end, :]
    u_prev = u_pred[:, 1:end-2, :]
    du_dt_pred = (u_next .- u_prev) ./ (2.0f0 * dt_step)
    
    # 2. Calculate true HH derivatives for the interior points
    u_interior = u_pred[:, 2:end-1, :]
    
    # Reshape from (4, N-2, 1) to (4, N-2) so your hh_equations handles it
    u_interior_2d = reshape(u_interior, 4, :)
    true_derivs_2d = hh_equations(u_interior_2d)
    
    # Reshape back to sequence shape (4, N-2, 1)
    true_derivs_seq = reshape(true_derivs_2d, 4, size(u_interior, 2), 1)
    
    # 3. Scale-aware physics residual
    residuals = (du_dt_pred .- true_derivs_seq) ./ s_factors
    
    return mean(abs2, residuals)
end

function total_loss(p, _)
    # Forward pass the sequence of time through the Transformer
    u_pred, _ = pinnsformer(t_seq, p, st)
    
    # Data loss: Mean Squared Error between prediction and actual sequence
    loss_data = mean(abs2, u_pred .- z_seq)
    
    # Physics loss: Enforce the HH dynamics on the predicted sequence
    loss_phys = physics_loss_seq(u_pred, dt, scale_factors)
    
    λ = 1.5f0
    return loss_data + λ * loss_phys
end

In [ ]:
# %%
loss_history = Float32[]

callback_fn = function (p, l)
    push!(loss_history, l)
    if length(loss_history) % 50 == 0
        println("Epoch $(length(loss_history)) | Total Loss: $(round(l, digits=5))")
    end
    return false # Continue training
end

# Setup optimization problem with AutoZygote
opt_func = OptimizationFunction(total_loss, Optimization.AutoZygote())
opt_prob = OptimizationProblem(opt_func, p_nn)

println("Starting Deterministic PINNsFormer Training...")

# Note: Learning rate is set to 0.0005 (Transformers can be sensitive)
result = solve(opt_prob, Adam(0.0005), maxiters = 3000, callback = callback_fn)

println("Training Complete!")

# Extract final optimized parameters
p_opt = result.u

# Plot the loss
using Plots
plot(loss_history, 
    title = "PINNsFormer Training Loss",
    xlabel = "Epochs", 
    ylabel = "Total Loss", 
    linewidth = 2, 
    color = :cyan,
    yscale = :log10, # Log scale is often better for viewing PINN convergence
    grid = true,
    label = "Loss (Data + Physics)"
)